# Phase M5 Validation: BraTS-PTG Video Generation
**Goal:** Validate DDPM trained on MU-Glioma on held-out BraTS-PTG data.
**Input:** `ddpm_v2_128_best.pth` + `brats_ptg_eval_triplets.npz`
**Output:** Dice table (DDPM vs Linear) + per-patient bar chart + visual comparison + videos
---

In [ ]:
import os,time,math,gc,warnings
import numpy as np
import torch,torch.nn as nn,torch.nn.functional as F
from pathlib import Path
from collections import defaultdict
from scipy import ndimage
warnings.filterwarnings('ignore')
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUT=Path('/kaggle/working/video_val');OUT.mkdir(parents=True,exist_ok=True)
NOISE_T=5;N_INTERP=15;FPS=3
PX_TO_ML=(240/128)**2*1.0/1000.0
print(f'Device: {device}')

In [ ]:
# Load BraTS-PTG eval triplets
tp=None
for p in Path('/kaggle/input').rglob('brats_ptg_eval_triplets.npz'):tp=p;break
if tp is None:raise RuntimeError('brats_ptg_eval_triplets.npz not found!')
d=np.load(tp,allow_pickle=True)
seg_s,seg_e,seg_g=d['seg_start'],d['seg_end'],d['seg_gt']
t1c_s,t_interp,pids=d['t1c_start'],d['t_interp'],d['pids']
N=len(seg_s)
unique_pids=np.unique(pids)
print(f'Loaded {N} BraTS-PTG triplets from {len(unique_pids)} patients')
for pid in unique_pids[:5]:print(f'  {pid}: {(pids==pid).sum()} triplets')

In [ ]:
# Noise schedule
T=50
def cosine_betas(T,s=0.008):
    t=torch.linspace(0,T,T+1)
    ac=torch.cos(((t/T)+s)/(1+s)*math.pi*0.5)**2
    ac=ac/ac[0];b=1-(ac[1:]/ac[:-1])
    return torch.clip(b,0.0001,0.9999)
betas=cosine_betas(T).to(device)
alphas=1-betas;ac=torch.cumprod(alphas,0)
sac=torch.sqrt(ac);s1mac=torch.sqrt(1-ac)

In [ ]:
# Model architecture
BASE_CH=96
class SinPE(nn.Module):
    def __init__(s,d):super().__init__();s.d=d
    def forward(s,t):
        h=s.d//2;e=math.log(10000)/(h-1)
        e=torch.exp(torch.arange(h,device=t.device)*-e)
        e=t[:,None].float()*e[None,:];return torch.cat([e.sin(),e.cos()],-1)
class FiLMResBlock(nn.Module):
    def __init__(s,ic,oc,cond_ch,td):
        super().__init__()
        g1=min(8,ic) if ic%8==0 else(4 if ic%4==0 else 1)
        g2=min(8,oc) if oc%8==0 else(4 if oc%4==0 else 1)
        s.c1=nn.Sequential(nn.GroupNorm(g1,ic),nn.SiLU(),nn.Conv2d(ic,oc,3,1,1))
        s.c2=nn.Sequential(nn.GroupNorm(g2,oc),nn.SiLU(),nn.Conv2d(oc,oc,3,1,1))
        s.tp=nn.Sequential(nn.SiLU(),nn.Linear(td,oc))
        s.film=nn.Conv2d(cond_ch,oc*2,1)
        s.sk=nn.Conv2d(ic,oc,1) if ic!=oc else nn.Identity()
    def forward(s,x,te,cf):
        h=s.c1(x);h=h+s.tp(te)[:,:,None,None]
        fs=s.film(cf);sc,sh=fs.chunk(2,dim=1)
        h=h*(1+sc)+sh;h=s.c2(h);return h+s.sk(x)
class CondEncoder(nn.Module):
    def __init__(s,ic=10,base=96):
        super().__init__()
        s.e1=nn.Sequential(nn.Conv2d(ic,base,3,1,1),nn.SiLU())
        s.e2=nn.Sequential(nn.Conv2d(base,base*2,3,2,1),nn.SiLU())
        s.e3=nn.Sequential(nn.Conv2d(base*2,base*4,3,2,1),nn.SiLU())
        s.e4=nn.Sequential(nn.Conv2d(base*4,base*8,3,2,1),nn.SiLU())
    def forward(s,x):
        f1=s.e1(x);f2=s.e2(f1);f3=s.e3(f2);f4=s.e4(f3);return[f1,f2,f3,f4]
class DDPMv2(nn.Module):
    def __init__(s,x_ch=5,cond_ch=10,B=96,td=256):
        super().__init__()
        s.t_emb=nn.Sequential(SinPE(td),nn.Linear(td,td),nn.SiLU())
        s.ti_emb=nn.Sequential(nn.Linear(1,td),nn.SiLU(),nn.Linear(td,td))
        s.cond_enc=CondEncoder(cond_ch,B)
        s.inp=nn.Sequential(nn.Conv2d(x_ch,B,3,1,1),nn.SiLU())
        s.e1=FiLMResBlock(B,B,B,td);s.e2=FiLMResBlock(B,B*2,B*2,td)
        s.e3=FiLMResBlock(B*2,B*4,B*4,td);s.e4=FiLMResBlock(B*4,B*8,B*8,td)
        s.dn=nn.MaxPool2d(2)
        s.up=nn.Upsample(scale_factor=2,mode='bilinear',align_corners=False)
        s.d3=FiLMResBlock(B*8+B*4,B*4,B*4,td)
        s.d2=FiLMResBlock(B*4+B*2,B*2,B*2,td)
        s.d1=FiLMResBlock(B*2+B,B,B,td)
        s.out=nn.Conv2d(B,x_ch,1)
    def forward(s,xn,td_,cond,ti):
        te=s.t_emb(td_)+s.ti_emb(ti)
        cf=s.cond_enc(cond)
        x=s.inp(xn)
        e1=s.e1(x,te,cf[0]);e2=s.e2(s.dn(e1),te,cf[1])
        e3=s.e3(s.dn(e2),te,cf[2]);e4=s.e4(s.dn(e3),te,cf[3])
        d3=s.d3(torch.cat([s.up(e4),e3],1),te,cf[2])
        d2=s.d2(torch.cat([s.up(d3),e2],1),te,cf[1])
        d1=s.d1(torch.cat([s.up(d2),e1],1),te,cf[0])
        return s.out(d1)
model=DDPMv2(x_ch=5,cond_ch=10,B=BASE_CH).to(device)
ckpt=None
for p in sorted(Path('/kaggle/input').rglob('ddpm_v2_128_v4_best.pth')):ckpt=p;break
if ckpt is None:raise RuntimeError('Checkpoint not found!')
ck=torch.load(ckpt,map_location=device,weights_only=False)
model.load_state_dict(ck['model']);model.eval()
print(f'Model: {sum(p.numel() for p in model.parameters())/1e6:.1f}M params')

In [ ]:
# Helpers
def seg_to_soft(seg,C=5):
    oh=np.zeros((C,seg.shape[0],seg.shape[1]),dtype=np.float32)
    for c in range(C):oh[c]=(seg==c).astype(np.float32)
    return oh*1.9-0.95
def predict_x0(xt,t,np_):
    return ((xt-s1mac[t][:,None,None,None]*np_)/sac[t][:,None,None,None].clamp(min=1e-4)).clamp(-2,2)
def enforce_containment(seg):
    out=seg.clone();tc=(out==1)|(out==4);et=out==4;out[et&~tc]=1;return out
def morpho_cleanup(seg):
    out=seg.copy()
    for label in [1,2,4]:
        mask=(out==label)
        if mask.sum()<5:continue
        closed=ndimage.binary_closing(mask,structure=np.ones((3,3)),iterations=2)
        wt=out>0;new_pixels=closed&(~mask)&wt;out[new_pixels]=label
    wt=out>0;labeled,n=ndimage.label(wt)
    for c in range(1,n+1):
        if(labeled==c).sum()<20:out[labeled==c]=0
    return out
@torch.no_grad()
def generate_frame(model,seg_start,seg_end,ti_val):
    ss=torch.from_numpy(seg_to_soft(seg_start)).unsqueeze(0).to(device)
    se=torch.from_numpy(seg_to_soft(seg_end)).unsqueeze(0).to(device)
    cond=torch.cat([ss,se],1);blend=(1-ti_val)*ss+ti_val*se
    t=torch.full((1,),NOISE_T,device=device,dtype=torch.long)
    xt=sac[t][:,None,None,None]*blend+s1mac[t][:,None,None,None]*torch.randn_like(blend)
    ti=torch.full((1,1),ti_val,device=device)
    return morpho_cleanup(enforce_containment(predict_x0(xt,t,model(xt,t,cond,ti)).argmax(1)[0]).cpu().numpy())
def dice_sc(p,g):
    r={}
    for n,pm,gm in[('WT',p>0,g>0),('TC',(p==1)|(p==4),(g==1)|(g==4)),('ET',p==4,g==4)]:
        r[n]=(2*(pm&gm).sum().float()/(pm.sum().float()+gm.sum().float()+1e-8)).item()
    return r
def label_rgb(s):
    r=np.zeros((*s.shape,3),dtype=np.float32)
    r[s==1]=[0.2,0.4,1.0];r[s==2]=[0.2,0.8,0.3];r[s==4]=[1.0,0.2,0.2];return r
def overlay_all(seg,t1c,alpha=0.6):
    tr=np.stack([t1c]*3,-1);sr=label_rgb(seg);m=sr.sum(-1,keepdims=True)>0
    return np.clip(np.where(m,(1-alpha)*tr+alpha*sr,tr),0,1)
print('Helpers ready')

In [ ]:
# === Dice Evaluation: DDPM vs Linear on ALL BraTS-PTG triplets ===
print(f'Evaluating {N} BraTS-PTG triplets...')
results_model = {'WT':[],'TC':[],'ET':[]}
results_linear = {'WT':[],'TC':[],'ET':[]}
per_patient = {}

for i in range(N):
    gt = seg_g[i]; ti = float(t_interp[i]); pid = str(pids[i])
    # Model prediction
    pred = generate_frame(model, seg_s[i], seg_e[i], ti)
    d = dice_sc(torch.tensor(pred), torch.tensor(gt))
    for k in d: results_model[k].append(d[k])
    # Linear baseline
    ss_soft = seg_to_soft(seg_s[i]); se_soft = seg_to_soft(seg_e[i])
    lin = morpho_cleanup(np.argmax((1-ti)*ss_soft + ti*se_soft, axis=0).astype(np.int64))
    dl = dice_sc(torch.tensor(lin), torch.tensor(gt))
    for k in dl: results_linear[k].append(dl[k])
    # Per patient
    if pid not in per_patient:
        per_patient[pid] = {'model':{'WT':[],'TC':[],'ET':[]}, 'linear':{'WT':[],'TC':[],'ET':[]}}
    for k in d:
        per_patient[pid]['model'][k].append(d[k])
        per_patient[pid]['linear'][k].append(dl[k])
    if (i+1) % 50 == 0: print(f'  {i+1}/{N} done')

# Print results table
print()
print('=' * 65)
print('  BraTS-PTG External Validation Results')
print('=' * 65)
header = f"{'Method':<20}{'WT':>10}{'TC':>10}{'ET':>10}{'Avg':>10}"
print(header)
print('-' * 65)
for name, res in [('DDPM (ours)', results_model), ('Linear interp', results_linear)]:
    wt = np.mean(res['WT']); tc = np.mean(res['TC']); et = np.mean(res['ET'])
    avg = np.mean([wt, tc, et])
    row = f'{name:<20}{wt:10.3f}{tc:10.3f}{et:10.3f}{avg:10.3f}'
    print(row)
print('=' * 65)
print(f'N={N} triplets from {len(unique_pids)} patients')

In [ ]:
# === Per-patient breakdown + comparison bar chart ===
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt

print('Per-Patient Dice (Model vs Linear)')
print('-' * 80)
for pid in sorted(per_patient.keys()):
    pp = per_patient[pid]
    n = len(pp['model']['WT'])
    wm = np.mean(pp['model']['WT']); wl = np.mean(pp['linear']['WT'])
    tm = np.mean(pp['model']['TC']); tl = np.mean(pp['linear']['TC'])
    em = np.mean(pp['model']['ET']); el = np.mean(pp['linear']['ET'])
    print(f'{pid:<20} n={n:3d}  WT:{wm:.3f}/{wl:.3f}  TC:{tm:.3f}/{tl:.3f}  ET:{em:.3f}/{el:.3f}')

# Bar chart
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('BraTS-PTG: DDPM vs Linear Interpolation (per patient)', fontsize=13, fontweight='bold')
colors_m, colors_l = '#1565C0', '#E65100'
sorted_pids = sorted(per_patient.keys())
for ax, metric in zip(axes, ['WT', 'TC', 'ET']):
    m_vals = [np.mean(per_patient[p]['model'][metric]) for p in sorted_pids]
    l_vals = [np.mean(per_patient[p]['linear'][metric]) for p in sorted_pids]
    x = np.arange(len(m_vals)); w = 0.35
    ax.bar(x - w/2, m_vals, w, label='DDPM', color=colors_m, alpha=0.8)
    ax.bar(x + w/2, l_vals, w, label='Linear', color=colors_l, alpha=0.8)
    ax.set_ylabel('Dice'); ax.set_title(f'{metric} Dice', fontweight='bold')
    ax.set_xticks(x); ax.set_xticklabels([p[-5:] for p in sorted_pids], rotation=45, fontsize=7)
    ax.legend(fontsize=8); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(str(OUT / 'brats_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved brats_comparison.png')

In [ ]:
# === Visual comparison: 6 BraTS-PTG samples ===
fig, axes = plt.subplots(6, 5, figsize=(20, 24))
fig.suptitle('BraTS-PTG Validation: Start | GT | DDPM | Linear | End', fontsize=14, fontweight='bold')
col_titles = ['Start', 'Ground Truth', 'DDPM Prediction', 'Linear Interp', 'End']
for j, c in enumerate(col_titles):
    axes[0, j].set_title(c, fontsize=11, fontweight='bold')

sample_idx = np.linspace(0, N-1, 6, dtype=int)
for row, si in enumerate(sample_idx):
    gt = seg_g[si]; t1c = t1c_s[si]; ti = float(t_interp[si])
    pred = generate_frame(model, seg_s[si], seg_e[si], ti)
    ss_soft = seg_to_soft(seg_s[si]); se_soft = seg_to_soft(seg_e[si])
    lin = morpho_cleanup(np.argmax((1-ti)*ss_soft + ti*se_soft, axis=0).astype(np.int64))
    dm = dice_sc(torch.tensor(pred), torch.tensor(gt))
    dl = dice_sc(torch.tensor(lin), torch.tensor(gt))
    for j, seg in enumerate([seg_s[si], gt, pred, lin, seg_e[si]]):
        axes[row, j].imshow(overlay_all(seg, t1c), origin='lower')
        axes[row, j].axis('off')
    axes[row, 2].set_xlabel(f"WT={dm['WT']:.2f} TC={dm['TC']:.2f} ET={dm['ET']:.2f}", fontsize=8)
    axes[row, 3].set_xlabel(f"WT={dl['WT']:.2f} TC={dl['TC']:.2f} ET={dl['ET']:.2f}", fontsize=8)
    axes[row, 0].set_ylabel(f"{pids[si]}\nt={ti:.2f}", fontsize=8, rotation=0, ha='right', va='center')

plt.tight_layout()
plt.savefig(str(OUT / 'brats_visual.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved brats_visual.png')

In [ ]:
# === Generate videos for ALL BraTS-PTG patients (4 per patient: ALL/WT/TC/ET) ===
try:
    import imageio; HAS_IMAGEIO = True
except: HAS_IMAGEIO = False

# Subregion helpers
def get_wt(seg): return (seg > 0).astype(np.float32)
def get_tc(seg): return ((seg == 1) | (seg == 4)).astype(np.float32)
def get_et(seg): return (seg == 4).astype(np.float32)

REGION_CFG = {
    'WT': {'fn': get_wt, 'color': np.array([0.2, 0.8, 0.3]), 'label': 'Whole Tumor'},
    'TC': {'fn': get_tc, 'color': np.array([0.2, 0.4, 1.0]), 'label': 'Tumor Core'},
    'ET': {'fn': get_et, 'color': np.array([1.0, 0.2, 0.2]), 'label': 'Enhancing Tumor'},
}

def overlay_region(mask, t1c, color, alpha=0.6):
    tr = np.stack([t1c] * 3, -1)
    sr = np.zeros_like(tr)
    sr[mask > 0.5] = color
    m = (mask > 0.5)[:, :, None]
    return np.clip(np.where(m, (1 - alpha) * tr + alpha * sr, tr), 0, 1)

# Build unique scan pairs per patient
brats_pairs = defaultdict(list)
seen = set()
for i in range(N):
    pid = str(pids[i])
    key = (pid, seg_s[i].tobytes()[:100], seg_e[i].tobytes()[:100])
    if key not in seen:
        seen.add(key)
        brats_pairs[pid].append({
            'seg_start': seg_s[i], 'seg_end': seg_e[i], 't1c_start': t1c_s[i]
        })

vid_dir = OUT / 'videos'; vid_dir.mkdir(exist_ok=True)
all_pids = sorted(brats_pairs.keys())
# Compute actual scan count from triplet count
# Video_A builds all C(N,3) triplets per patient, so C(N,3) = n_triplets
# Invert: find N where N*(N-1)*(N-2)/6 = n_triplets
def triplets_to_scans(n_trip):
    for n in range(3, 30):
        if n * (n-1) * (n-2) // 6 == n_trip:
            return n
    return 3  # fallback

triplet_counts = defaultdict(int)
for i in range(N):
    triplet_counts[str(pids[i])] += 1

scan_counts = {}
for pid in all_pids:
    scan_counts[pid] = triplets_to_scans(triplet_counts[pid])
    
# Also build CONSECUTIVE pairs from ordered unique scans per patient
# Use full array hash to find unique scans
from hashlib import md5
patient_scans = defaultdict(dict)  # pid -> {hash: seg_array}
patient_t1cs = {}
for i in range(N):
    pid = str(pids[i])
    for seg_arr in [seg_s[i], seg_e[i]]:
        h = md5(seg_arr.tobytes()).hexdigest()
        if h not in patient_scans[pid]:
            patient_scans[pid][h] = seg_arr
    if pid not in patient_t1cs:
        patient_t1cs[pid] = t1c_s[i]

# Build consecutive pairs from the unique scans (ordered by their position)
brats_pairs = defaultdict(list)
for pid in all_pids:
    unique_list = list(patient_scans[pid].values())
    if len(unique_list) < 2: continue
    for j in range(len(unique_list) - 1):
        brats_pairs[pid].append({
            'seg_start': unique_list[j],
            'seg_end': unique_list[j + 1],
            't1c_start': patient_t1cs[pid],
        })
for sc in sorted(set(scan_counts.values())):
    (vid_dir / f'{sc}_scans').mkdir(exist_ok=True)
    n = sum(1 for v in scan_counts.values() if v == sc)
    print(f'  {sc} scans: {n} patients')
print(f'Generating videos for ALL {len(all_pids)} BraTS-PTG patients (4 videos each)...')

for pi, pid in enumerate(all_pids):
    pairs = brats_pairs[pid][:4]  # max 4 gaps
    t1c = pairs[0]['t1c_start']

    # Generate all segmentation frames
    all_segs = []
    for gi, pair in enumerate(pairs):
        ss = pair['seg_start']; se = pair['seg_end']
        for fi, tv in enumerate(np.linspace(0, 1, N_INTERP)):
            if tv == 1.0 and gi < len(pairs) - 1: continue
            if tv == 0: seg = morpho_cleanup(ss)
            elif tv == 1.0: seg = morpho_cleanup(se)
            else: seg = generate_frame(model, ss, se, tv)
            all_segs.append((seg, f'Scan {gi+1}->{gi+2} | t={tv:.2f}'))

    # Build 4 video types
    for vid_type in ['ALL', 'WT', 'TC', 'ET']:
        frames = []
        for seg, lbl in all_segs:
            if vid_type == 'ALL':
                img = overlay_all(seg, t1c)
                wt_ml = get_wt(seg).sum() * PX_TO_ML
                tc_ml = get_tc(seg).sum() * PX_TO_ML
                et_ml = get_et(seg).sum() * PX_TO_ML
                title = f'{pid}\n{lbl}\nWT={wt_ml:.1f}mL  TC={tc_ml:.1f}mL  ET={et_ml:.1f}mL'
            else:
                cfg = REGION_CFG[vid_type]
                mask = cfg['fn'](seg)
                img = overlay_region(mask, t1c, cfg['color'])
                vol_ml = mask.sum() * PX_TO_ML
                title = f'{pid} [{cfg["label"]}]\n{lbl}\nVolume: {vol_ml:.2f} mL'
            fig, ax = plt.subplots(1, 1, figsize=(5, 5), dpi=100)
            ax.imshow(img, origin='lower'); ax.axis('off')
            ax.set_title(title, fontsize=9, fontweight='bold', pad=6)
            fig.tight_layout(pad=0.3); fig.canvas.draw()
            buf = np.array(fig.canvas.buffer_rgba())[:, :, :3].copy()
            frames.append(buf); plt.close(fig)

        if HAS_IMAGEIO and frames:
            sc = scan_counts[pid]
            fp = str(vid_dir / f'{sc}_scans' / f'{pid}_{vid_type}.mp4')
            imageio.mimsave(fp, frames, fps=FPS)

    if (pi + 1) % 10 == 0 or pi == 0:
        print(f'  [{pi+1}/{len(all_pids)}] {pid}: {len(all_segs)} frames x 4 videos')

print(f'Done! {len(all_pids) * 4} videos saved to {vid_dir}')

In [ ]:
# Final summary
print()
print('=' * 65)
print('  PHASE M5 VALIDATION: BraTS-PTG - COMPLETE')
print('=' * 65)
wm = np.mean(results_model['WT']); tm = np.mean(results_model['TC']); em = np.mean(results_model['ET'])
wl = np.mean(results_linear['WT']); tl = np.mean(results_linear['TC']); el = np.mean(results_linear['ET'])
print(f'  BraTS-PTG triplets:  {N} from {len(unique_pids)} patients')
print(f'  DDPM Dice:    WT={wm:.3f}  TC={tm:.3f}  ET={em:.3f}  avg={np.mean([wm,tm,em]):.3f}')
print(f'  Linear Dice:  WT={wl:.3f}  TC={tl:.3f}  ET={el:.3f}  avg={np.mean([wl,tl,el]):.3f}')
print(f'  Model trained on:  MU-Glioma (N=152)')
print(f'  Tested on:         BraTS-PTG (external, unseen)')
print(f'  Files:')
for f in sorted(OUT.rglob('*')):
    if f.is_file(): print(f'    {f.relative_to(OUT)}  {f.stat().st_size/1e6:.1f}MB')
print('=' * 65)